In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)

import sys
sys.path.append("/ihome/ylee/yiz133/Code/Data processing/functions")
import importlib
import functions.mdata_utils as mdata_utils
import functions.TCR_embedings as TCR_embedings

2026-05-20 00:21:05.122272: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 00:21:05.166520: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 00:21:06.480147: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-20 00:21:12.767092: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different compu

ModuleNotFoundError: No module named 'functions'

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA

In [ ]:
importlib.reload(TCR_embedings)

In [ ]:
path = '/ix1/ylee/Yifan_Zhang/Code_data/data_EAE/processed/'
filename = "EAE_HV_then_merged_testGSELEE.h5mu"
mdata_ori = mu.read(path + filename)

In [ ]:
mdata = mdata_ori.copy()
mdata

In [ ]:
aa

In [ ]:
mdata['airr'].obs['GSE'] = mdata.obs['GSE']
mdata['airr'].obs['GSE'].value_counts()

# Settings

In [ ]:
# mdata_sets = {'all': mdata,
#             'cloned': mdata_cloned,
#             'one_in_each_clone': mdata_cloned_oneCell}
# mdata_sub = mdata_sets['cloned']

### Sets to use
subset_by_GSE = False
if subset_by_GSE == True:
    target_set = ['GSE188320', 'GSE178085']
    mdata = mdata[mdata.obs['GSE'].isin(target_set)]
    
    # redo harmony in the subset
    import scanpy.external as sce
    sc.pp.pca(mdata["gex"], n_comps=50)
    sc.pp.neighbors(mdata["gex"], n_neighbors = 50)
    sc.tl.umap(mdata["gex"], min_dist=0.5, spread=1.0)
    sce.pp.harmony_integrate(mdata["gex"], key=["GSE", 'sample_id'],  basis='X_pca',
                          theta = 3, lamb = 1,  sigma=0.1, nclust = 50, tau=1)


### If test group is isolated
if_test_isolated = True
if if_test_isolated == True:
    test_ids = ['GSE293883']
else:
    train_frac = 0.7
    
### If use x_umap of X_umap_harmony
Batch_corrected = True

mdata    

# data pre-vision

In [ ]:
mdata['gex'].obs['tissue'].isna().sum()

In [ ]:
# mdata['gex'].obs['Tissue_group'] = mdata['gex'].obs['tissue'].apply(lambda x: x if (x == 'CNS' or x =='Spleen') else 'else')

mdata['gex'].obs['Tissue_group'] = mdata['gex'].obs['tissue'].apply(lambda x: x if (x == 'CNS') else 'non-CNS')
mdata['gex'].obs['Tissue_group'].value_counts()

In [ ]:
mdata['gex'].obs['Tissue_group'].isna().sum()

In [ ]:
sc.pl.umap(mdata["gex"], color=['state', 'tissue', 'cell_type', 'GSE'], ncols=2, wspace=0.5)

In [ ]:
gene_PCA = 'X_pca_harmony' if Batch_corrected else 'X_pca'
gene_Umap = 'X_umap_harmony' if Batch_corrected else 'X_umap'
neighbors_key = 'neighbors_harmony' if Batch_corrected else 'neighbors'

sc.pl.embedding(mdata["gex"], basis = gene_Umap, neighbors_key = neighbors_key, 
                color=['GSE', 'state', 'Tissue_group', 'cell_type'], ncols=2, wspace=0.2)

# Subsets

#### Cell type subset

In [ ]:
# print(mdata.obs['condition'].value_counts())
# print(mdata.obs['cell_type'].value_counts())
# print(mdata['gex'].obs['tissue'].value_counts())

In [ ]:
# mdata = mdata[mdata.obs['condition'].isin(['EAE'])]
# mdata = mdata[~mdata.obs['cell_type'].isin(['CD8'])]

# Suppose mdata['gex'].obs has columns 'state' and 'cell_type'
# cols = ['state', 'cell_type']
# mask = mdata['gex'].obs[cols].notna().all(axis=1)
# mdata = mdata[mask].copy()

In [ ]:
mdata

#### only cloned TCRs

In [ ]:
min_clone = 1
cloned_mask = mdata['airr'].obs['clone_id_size'].fillna(0).astype('int') > min_clone
mdata_cloned = mdata[cloned_mask].copy()
mdata_single = mdata[~cloned_mask].copy()
mdata_cloned['airr'].obs['GSE'].value_counts()

#### select only one cell from each clonotype

In [ ]:
# Randomly keep one cell per clone_id within each GSE subset
airr = mdata_cloned['airr']
# airr = mdata['airr']
print(f"Original number of cells: {airr.n_obs}")

# Store indices to keep
oneCell_per_clone_indices = []

# Group by GSE
for gse in airr.obs['GSE'].unique():
    # Get cells for this GSE
    gse_mask = airr.obs['GSE'] == gse
    gse_obs = airr.obs[gse_mask]
    
    print(f"\nGSE: {gse} - Original cells: {len(gse_obs)}")
    
    # For each clone_id in this GSE, randomly select one cell
    for clone_id in gse_obs['clone_id'].dropna().unique():
        clone_cells = gse_obs[gse_obs['clone_id'] == clone_id]
        if len(clone_cells) > 1:
        # Randomly sample one cell from this clone
            selected_idx = clone_cells.sample(n=1, random_state=42).index[0]
            oneCell_per_clone_indices.append(selected_idx)
        else:
            oneCell_per_clone_indices.append(clone_cells.index[0])
    
    print(f"GSE: {gse} - After sampling: {len(gse_obs['clone_id'].unique())} cells (one per clone)")

# Create subset with selected cells
mdata_cloned_oneCell = mdata[oneCell_per_clone_indices].copy()


## select subset

In [ ]:
single_per_clone = list(mdata_single.obs['GSE'].index) + oneCell_per_clone_indices
mdata_single_withCloned = mdata[single_per_clone].copy()

mdata = mdata_single
mdata

# embed TCR AA into vector

In [ ]:
# beta chain only
# tcr_aa_obs = ['VDJ_1_cdr3_aa',]
# tcr_cat_features = ['VDJ_1_j_call', 'VDJ_1_v_call']

# both chains
tcr_aa_obs = ['VDJ_1_cdr3_aa', 'VJ_1_cdr3_aa']
tcr_cat_features = ['VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call']

tcr_num_features = []

In [ ]:
# Atchley factors for the 20 amino acids
atchley_factors = {
    'A': [ 0.591, -1.302, -0.733,  1.570, -0.146],  # Alanine
    'R': [ 1.538,  0.055,  1.502,  0.440,  2.897],  # Arginine
    'N': [ 0.945,  0.828,  1.299, -0.169,  0.933],  # Asparagine
    'D': [ 1.050,  0.302, -3.656, -0.259, -3.242],  # Aspartic acid
    'C': [-1.343,  0.465, -0.862, -1.020, -0.255],  # Cysteine
    'Q': [ 0.931,  0.179, -3.005, -0.503, -1.853],  # Glutamine
    'E': [ 1.357,  0.113, -3.242, -0.339, -2.192],  # Glutamic acid
    'G': [ 0.384,  1.652,  1.330,  1.045,  2.064],  # Glycine
    'H': [ 0.336, -0.417, -1.673, -1.474, -0.078],  # Histidine
    'I': [-1.239, -0.547,  2.131,  0.393,  0.816],  # Isoleucine
    'L': [-1.019, -0.987, -1.505,  1.266, -0.912],  # Leucine
    'K': [ 1.831, -0.561,  0.533, -0.277,  1.648],  # Lysine
    'M': [-0.663, -1.524,  2.219, -1.005,  1.212],  # Methionine
    'F': [-1.006, -0.590,  1.891, -0.397,  0.412],  # Phenylalanine
    'P': [ 0.189,  2.081, -1.628,  0.421, -1.392],  # Proline
    'S': [ 0.228,  1.399, -4.760,  0.670, -2.647],  # Serine
    'T': [ 0.032,  2.213, -1.455,  0.311, -0.259],  # Threonine
    'W': [-0.595,  0.009,  0.672, -2.128, -0.184],  # Tryptophan
    'Y': [ 0.260,  0.830,  3.097, -0.838,  1.512],  # Tyrosine
    'V': [-1.337, -0.279, -0.544,  1.242, -1.262],  # Valine
}


In [ ]:
# Vectorize each TCR amino acid column and compute composition & length
for aa_col in tcr_aa_obs:
    # 1. Sequence length
    # Only keep cells with both alpha and beta chains
    lengths = TCR_embedings.compute_sequence_lengths(mdata, aa_col)
    lengths_mask = (lengths> 10) & (lengths< 20)
    len_key = f'{aa_col}_length'
    mdata = mdata[lengths_mask]
    mdata.obs[len_key] = lengths[lengths_mask]
    print(f"Stored {aa_col} lengths in mdata.obs['{len_key}']")
    
    # 2. Atchley factor encoding
    encoded = TCR_embedings.vectorize_tcr_column(mdata, aa_col, atchley_factors)
    key_name = f'X_{aa_col}_atchley'
    mdata.obsm[key_name] = encoded
    print(f"Stored {aa_col} Atchley vectors in mdata.obsm['{key_name}'] with shape {encoded.shape}")
    
    # 2b. Adjacent Atchley factor interactions
    atchley_positions = encoded.shape[1] // 5
    pairwise_key = f'X_{aa_col}_atchley_pairwise'
    if atchley_positions > 1:
        encoded_reshaped = encoded.reshape(encoded.shape[0], atchley_positions, 5)
        pairwise_features = []
        for pos in range(atchley_positions - 1):
            current = encoded_reshaped[:, pos, :]
            nxt = encoded_reshaped[:, pos + 1, :]
            outer = (current[:, :, None] * nxt[:, None, :]).reshape(encoded.shape[0], -1)
            pairwise_features.append(outer)
        pairwise_matrix = np.concatenate(pairwise_features, axis=1)
    else:
        pairwise_matrix = np.zeros((encoded.shape[0], 0))
    mdata.obsm[pairwise_key] = pairwise_matrix
    print(f"Stored {aa_col} adjacent Atchley interactions in mdata.obsm['{pairwise_key}'] with shape {pairwise_matrix.shape}")    
    
    # 3. AA composition (percentage of each of 20 AAs)
    aa_comp = TCR_embedings.compute_aa_composition_matrix(mdata, aa_col)
    comp_key = f'X_{aa_col}_composition'
    mdata.obsm[comp_key] = aa_comp
    print(f"Stored {aa_col} AA composition in mdata.obsm['{comp_key}'] with shape {aa_comp.shape}\n")
    

In [ ]:
mdata

In [ ]:
# One-hot encode all categorical TCR features and concatenate
cat_encoded_list = []
for cat_col in tcr_cat_features:
    encoded = TCR_embedings.onehot_encode_categorical(mdata, cat_col)
    # cat_encoded_list.append(encoded)
    mdata.obsm[cat_col] = encoded

# Concatenate all one-hot encoded vectors
# tcr_cat_onehot = np.concatenate(cat_encoded_list, axis=1)


In [ ]:
mdata.obsm

In [ ]:
mdata.obsm['VDJ_1_v_call']

## concate TCR features

In [ ]:
arrs_tcr = []
for key, value in mdata.obsm.items():
    arrs_tcr.append(value)

In [ ]:
view_tcr = np.concatenate(arrs_tcr, axis=1)
print(view_tcr.shape)

# Add chain length
for chain in tcr_aa_obs:
    view_tcr = np.concatenate([view_tcr, mdata.obs[chain + '_length'].to_numpy().reshape(-1, 1)],  axis=1)

# Add clone size
view_tcr = np.concatenate([view_tcr, mdata['airr'].obs['clone_id_size'].to_numpy().reshape(-1, 1)],  axis=1)
    
print(view_tcr.shape)

In [ ]:
### Use DEGs for cca  #####
# top_n = 1000
# sc.tl.rank_genes_groups(mdata["gex"], groupby='tissue', groups = ['CNS'], reference='rest',
#                         n_genes = top_n, method='t-test')
# rank_df = sc.get.rank_genes_groups_df(mdata["gex"], None)


# mdata_HV = mdata["gex"][:, mdata["gex"].var_names.isin(rank_df['names'])]
# gex_df = mdata_HV.to_df()
# mdata_new = mu.MuData({'gex': mdata_HV, 'airr': mdata['airr']})
# view_gene = gex_df

In [ ]:
# Perform Canonical Correlation Analysis separately on train and test sets
view_gene = mdata['gex'].X.toarray()
# view_gene = mdata['gex'].obsm['X_pca_harmony']

scaler = StandardScaler()
view_tcr = scaler.fit_transform(view_tcr)
view_gene = scaler.fit_transform(view_gene)

In [ ]:
mdata.obsm['tcr_embs'] = view_tcr

In [ ]:
# import anndata as ad
# ad.settings.allow_write_nullable_strings = True
# mdata.write(filename+'_atchleyEmbs.h5mu')

# single test

## train test split by samples

In [ ]:
# select data sets as test 

if 'set' not in mdata.obs.columns:
    if if_test_isolated:
        mdata.obs.loc[mdata.obs['GSE'].isin(test_ids), 'set'] = 'test'
        mdata.obs.loc[~mdata.obs['GSE'].isin(test_ids), 'set'] = 'train'

        # Split data based on mdata.obs['set']
        train_mask = mdata.obs['set'] == 'train'
        test_mask = mdata.obs['set'] == 'test'

    else:
        # generate boolean mask
        train_mask = np.random.rand(mdata.n_obs) < train_frac
        test_mask = ~train_mask

        mdata.obs.loc[train_mask, 'set'] = 'test'
        mdata.obs.loc[test_mask, 'set'] = 'train'
else:
    train_mask = mdata.obs['set'] == 'train'
    test_mask = mdata.obs['set'] == 'test'


In [ ]:
print(mdata.obs['set'].value_counts())
print(mdata[test_mask].obs['GSE'].value_counts())

In [ ]:
view_gene_train = view_gene[train_mask]
view_gene_test = view_gene[test_mask]
view_tcr_train = view_tcr[train_mask]
view_tcr_test = view_tcr[test_mask]

print(mdata[mdata.obs['set']=='test']['gex'].obs['tissue'].value_counts())

## Unbiased CCA

In [ ]:
importlib.reload(TCR_embedings)

In [ ]:
dim_cca = 3

In [ ]:
# Self CCA function
#  Fit CCA on train set
# cca = CCA(n_components=dim_cca)
# cca = TCR_embedings.rCCA(n_components=dim_cca, alpha_x=1, alpha_y=1,)

# view_gene_c_train, view_tcr_c_train = cca.fit_transform(view_gene_train, view_tcr_train)
# # Transform test set using fitted CCA
# view_gene_c_test, view_tcr_c_test = cca.transform(view_gene_test, view_tcr_test)

In [ ]:
from cca_zoo.models import rCCA
from cca_zoo.model_selection import GridSearchCV

# c1 = [0.1, 0.3, 0.7, 0.9]
# c2 = [0.1, 0.3, 0.7, 0.9]
# param_grid = {'c': [c1, c2]}
# cv = 2
c1 = [0.7]
c2 = [0.1]
param_grid = {'c': [c1, c2]}
cv = 2

# Custom scoring function
def scorer(estimator, X):
    dim_corrs = estimator.score(X)
    return dim_corrs.mean()
    
ridge = GridSearchCV(rCCA(latent_dims=dim_cca), param_grid=param_grid,
                     cv=cv, verbose=True, scoring=scorer).fit((view_gene_train, view_tcr_train)).best_estimator_

# cca = ridge


In [ ]:
# Get transformed data (canonical variates)
view1_transformed, view2_transformed = ridge.transform((view_gene_train, view_tcr_train))

# Calculate correlation scores for each dimension
correlations = ridge.score((view_gene_train, view_tcr_train))

print(f"Best regularization parameters: c = {ridge.c}")
print(f"Correlations per dimension: {correlations}")
print(f"Mean correlation: {correlations.mean():.4f}")

In [ ]:
cca = rCCA(latent_dims=dim_cca, c=[0.7, 0.1])
view_gene_c_train, view_tcr_c_train = cca.fit_transform((view_gene_train, view_tcr_train))
view_gene_c_test, view_tcr_c_test = cca.transform((view_gene_test, view_tcr_test))


In [ ]:
view_gene.shape

In [ ]:

print(f"\nTrain canonical variates shape: {view_gene_c_train.shape}")
print(f"Test canonical variates shape: {view_gene_c_test.shape}")

view_gene_c_train = scaler.fit_transform(view_gene_c_train)
view_tcr_c_train = scaler.fit_transform(view_tcr_c_train)
view_gene_c_test = scaler.fit_transform(view_gene_c_test)   
view_tcr_c_test = scaler.fit_transform(view_tcr_c_test)

# Combine for full dataset storage

view_gene_c = np.zeros((mdata.n_obs, dim_cca))
view_tcr_c = np.zeros((mdata.n_obs, dim_cca))
view_gene_c[train_mask] = view_gene_c_train
view_gene_c[test_mask] = view_gene_c_test
view_tcr_c[train_mask] = view_tcr_c_train
view_tcr_c[test_mask] = view_tcr_c_test

In [ ]:
# cca_zoo weights
cca.x_weights_ = cca.weights[0]
cca.y_weights_ = cca.weights[1]

In [ ]:
print(cca.x_weights_.shape)
print(cca.y_weights_.shape)
mdata.uns['cca_x_weights'] = cca.x_weights_
mdata.uns['cca_y_weights'] = cca.y_weights_

In [ ]:
# the linear correlation coefficients between an original variable and its corresponding canonical variate
# print(cca.x_loadings_.shape)
# print(cca.y_loadings_.shape)

In [ ]:
df_cca_weights = pd.DataFrame(cca.y_weights_)
# df_cca_weights.to_csv(filename + 'TCR_weights.csv')

In [ ]:
cca_weights_abs = df_cca_weights.abs()

cca_weights_abs_beta = cca_weights_abs.iloc[0:596].sum() + cca_weights_abs.iloc[1191:1128].sum()
cca_weights_abs_alpha = cca_weights_abs.iloc[596:596+596].sum() + cca_weights_abs.iloc[1128:1374].sum()
df_cca_weights_abs_sum = pd.DataFrame(
    [cca_weights_abs_beta, cca_weights_abs_alpha],
    index=['beta', 'alpha']
)

df_cca_weights_abs_sum = df_cca_weights_abs_sum / df_cca_weights_abs_sum.sum(axis=0)
df_cca_weights_abs_sum



In [ ]:
a = cca_weights_abs.iloc[0:596].sum()
a

In [ ]:
# save CV score in mdata
cv_list = []
for i in range(view_gene_c_train.shape[1]):
    mdata['gex'].obs['CV_score_'+str(i)] = view_tcr_c[:, i]
    cv_list.append('CV_score_'+str(i))

### Deep CCA

In [ ]:
from cca_zoo_new.deep import DCCA, architectures
from cca_zoo_new.deep.data import NumpyDataset, check_dataset, get_dataloaders
import lightning.pytorch as pl
from lightning import seed_everything
from matplotlib import pyplot as plt
from cca_zoo_new.deep import (
    DCCA,
    DCCA_EY,
    DCCA_NOI,
    DCCA_SDL,
    BarlowTwins,
    VICReg,
    architectures,
)
from cca_zoo_new.visualisation import (
    ScoreScatterDisplay,
    UMAPScoreDisplay,
    TSNEScoreDisplay,
)

train_dataset = NumpyDataset([view_gene_train, view_tcr_train])
test_dataset = NumpyDataset([view_gene_test, view_tcr_test])

train_loader = get_dataloaders(train_dataset, batch_size=4)
test_loader = get_dataloaders(test_dataset, batch_size=4)

LATENT_DIMS = dim_cca
EPOCHS = 2

encoder_1 = architectures.Encoder(latent_dimensions=LATENT_DIMS, feature_size=view_gene_train.shape[1])
encoder_2 = architectures.Encoder(latent_dimensions=LATENT_DIMS, feature_size=view_tcr_train.shape[1])

dcca = DCCA(latent_dimensions=LATENT_DIMS, encoders=[encoder_1, encoder_2])
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    enable_checkpointing=True,
    enable_model_summary=True,
    enable_progress_bar=True,
)
trainer.fit(dcca, train_loader)

In [ ]:
view_gene_c_train, view_tcr_c_train = dcca.transform(train_loader)
view_gene_c_test, view_tcr_c_test = dcca.transform(test_loader)

In [ ]:
# score_display = ScoreScatterDisplay.from_estimator(
#     dcca, test_loader, )
# score_display.plot(title="Deep CCA")
# plt.show()

## Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(view_gene_c_test[:, 0], view_tcr_c_test[:, 0], alpha=0.5, label='CC 1')
plt.scatter(view_gene_c_test[:, 1], view_tcr_c_test[:, 1], alpha=0.5, label='CC 2')
plt.xlabel('view_gene_c_test')
plt.ylabel('view_tcr_c_test')
# Set ylim to cover the central 95% of tcr canonical variates
y_95 = np.percentile(view_tcr_c_test, [2.5, 97.5])
plt.ylim(y_95)
plt.title('Scatter plot of CCA projected test sets')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
corr_train, corr_test = TCR_embedings.train_test_corr(view_gene_c_train, view_tcr_c_train, view_gene_c_test, view_tcr_c_test)

In [ ]:
ax = TCR_embedings.plot_train_test_corr(corr_train, corr_test)
ax.set_ylim(0, 1)

### visualize CV scores

In [ ]:
sc.pl.embedding(mdata["gex"], basis = gene_Umap, neighbors_key = neighbors_key, 
            color= cv_list, ncols=2,)

### High-score group

In [ ]:
cutoff = 1
highlight_list = []
for i in range(0, dim_cca):
    cv_name = f'CV_score_{i}'
    tmp_col = cv_name + "_high"
    highlight_list.append(tmp_col)
    
    vals = mdata['gex'].obs[cv_name].copy()
    mdata['gex'].obs[tmp_col] = np.where(vals > cutoff, vals, np.nan)
    mdata['gex'].obs[cv_name + "_high_cat"] = np.where(vals > cutoff, True, False)
    mdata['gex'].obs[cv_name + "_high_cat"] = mdata['gex'].obs[cv_name + "_high_cat"].astype('category')
    mdata['gex'].obs[cv_name + "_high_cat"] = mdata['gex'].obs[cv_name + "_high_cat"].cat.remove_unused_categories()


sc.pl.embedding(mdata["gex"], basis = gene_Umap, neighbors_key = neighbors_key, 
            color= highlight_list, ncols=2,   vmin=cutoff, vmax=5)


mdata['gex'].obs["CV_score_0_high_cat"].value_counts()

In [ ]:
m = mdata[mdata['gex'].obs["CV_score_2_high_cat"]==True]
m.obs['GSE'].value_counts()

In [ ]:
# what does each high-score cluster consists of
tissue_mask = mdata['gex'].obs['Tissue_group'].isin(['CNS'])
mdata_sub = mdata[test_mask]

all_results = []
abs_target = ['Tissue_group', 'cell_type', 'state']

# Precompute the overall (baseline) pool within the same tissue filter

df_CV = TCR_embedings.subset_fractions_in_CV_scores(dim_cca, mdata, abs_target, cutoff)
df_CV

In [ ]:
TCR_embedings.plot_subset_fractions_in_CV_scores(df_CV)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------
# obs = mdata_sub['gex'].obs.copy()
obs = mdata['gex'].obs.copy()

cv_cols = [f'CV_score_{i}' for i in range(dim_cca)]
df_long = obs.melt(id_vars='Tissue_group',
                   value_vars=cv_cols,
                   var_name='CV_component',
                   value_name='CV_score')

# ------------------------------------------------------------
# Violin plot
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))
ax = sns.violinplot(
    data=df_long,
    x='CV_component',
    y='CV_score',
    hue='Tissue_group',
    split=True,
    inner='box',
    palette={'CNS': 'tab:blue', 'non-CNS': 'tab:orange'},
)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.title("t-test of CV Scores (CNS vs non-CNS)")
plt.xlabel('Canonical Variate (CV)')
plt.ylabel('CV Score')
plt.legend(title='Tissue', frameon=False)

# ------------------------------------------------------------
# Student's t-test (unpaired)
# ------------------------------------------------------------
def cohens_d(x1, x2):
    x1, x2 = np.asarray(x1), np.asarray(x2)
    n1, n2 = len(x1), len(x2)
    s1, s2 = x1.std(ddof=1), x2.std(ddof=1)

    # pooled standard deviation
    s_pooled = np.sqrt(((n1 - 1)*s1**2 + (n2 - 1)*s2**2) / (n1 + n2 - 2))

    d = (x1.mean() - x2.mean()) / s_pooled
    return d

p_values = {}
cohen_d = {}
for cv in cv_cols:
    sub = df_long[df_long['CV_component'] == cv]
    cns_scores = sub.loc[sub['Tissue_group'] == 'CNS', 'CV_score']
    non_scores = sub.loc[sub['Tissue_group'] == 'non-CNS', 'CV_score']
    stat, p = ttest_ind(cns_scores, non_scores, equal_var=True, nan_policy='omit')
    p_values[cv] = p
    cohen_d[cv] = cohens_d(cns_scores, non_scores)
    
# ------------------------------------------------------------
# Annotate p-values above violins
# ------------------------------------------------------------
y_min, y_max = df_long['CV_score'].min(), df_long['CV_score'].max()
offset = (y_max - y_min) * 0.05

for i, cv in enumerate(cv_cols):
    p = p_values.get(cv)
    label = f"p = {p:.2e}" if p is not None else "NA"
    ax.text(i, y_max + offset, label,
            ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='black')

plt.tight_layout()
plt.show()


In [ ]:
cohen_d

### correlations

### High cv score HVGs VS tissue specific HVGs

In [ ]:
import seaborn as sns
top_n = 20
cv_obs_name = [f'CV_score_{i}' for i in range(dim_cca)]
top_cv_genes = []

# ,dim_cca
for i in range(0,dim_cca):
      cv_name = f'CV_score_{i}'
      tmp_col = cv_name + "_high_cat"

      sc.tl.rank_genes_groups(mdata["gex"], groupby= tmp_col, reference='rest',
                        n_genes = top_n, method='logreg')

      rank_df = sc.get.rank_genes_groups_df(mdata["gex"], None)

      sc.pl.rank_genes_groups_dotplot(mdata["gex"], n_genes=top_n, groupby=tmp_col)
      # marker_genes = rank_df['names'].to_list()
      # top_cv_genes.append(marker_genes)
      # corr_matrix_genes = TCR_embedings.pearson_corr(mdata['gex'], cv_obs_name, marker_genes, var_type='gene')

      # plt.figure(figsize=(8, 3))
      # sns.heatmap(corr_matrix_genes, annot=False, cmap='coolwarm', center=0, 
      #             vmin=-1, vmax=1, fmt='.3f', cbar_kws={'label': 'Pearson correlation'})

In [ ]:
m2 = mdata[(mdata['gex'].obs["CV_score_0_high_cat"]==True) ]
m2.obs['GSE'].value_counts()

### generate a tissue gene set score

In [ ]:
top_n = 20
rg_method = 'wilcoxon'
sc.tl.rank_genes_groups(mdata["gex"], groupby='tissue', groups = ['CNS'], reference='rest',
                        n_genes = top_n, method=rg_method)
rank_df = sc.get.rank_genes_groups_df(mdata["gex"], None)
rank_df.head(5)
top_CNS_genes = rank_df['names'].to_list()

# mdata_HV = mdata["gex"][:, mdata["gex"].var_names.isin(rank_df['names'])]
# gex_df = mdata_HV.to_df()
# mdata_new = mu.MuData({'gex': mdata_HV, 'airr': mdata['airr']})
# view_gene = gex_df

In [ ]:
CNS_HVGs = rank_df['names'].to_list()
sc.tl.score_genes(mdata['gex'], gene_list=CNS_HVGs, score_name='CNS_score')

In [ ]:
sc.tl.rank_genes_groups(mdata["gex"], groupby='tissue', groups = ['Spleen'], reference='rest',
                        n_genes = top_n, method=rg_method)
rank_df_spl = sc.get.rank_genes_groups_df(mdata["gex"], None)

SPL_HVGs = rank_df_spl['names'].to_list()
sc.tl.score_genes(mdata['gex'], gene_list=SPL_HVGs, score_name='SPL_score')

In [ ]:
# Create the heatmap
cv_obs_name = [f'CV_score_{i}' for i in range(dim_cca)]
scores = ['CD4score', 'CD8score', 'Tregscore', 'Th17score',
          'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore',
          'CNS_score', 'SPL_score']

# scores = ['CNS_score', 'SPL_score']
corr_matrix_obs = TCR_embedings.pearson_corr(mdata['gex'], cv_obs_name, scores, var_type='obs')


In [ ]:
# high_score_mask = (mdata['gex'].obs["CV_score_0_high_cat"]==True) | (mdata['gex'].obs["CV_score_1_high_cat"]==True)| (mdata['gex'].obs["CV_score_2_high_cat"]==True)
# corr_matrix_obs = TCR_embedings.pearson_corr(mdata[high_score_mask]['gex'], cv_obs_name, scores, var_type='obs')


In [ ]:
plt.figure(figsize=(8, 3))
# Reorder columns of the correlation matrix to match the order of 'scores'
corr_matrix_obs = corr_matrix_obs[scores]

sns.heatmap(corr_matrix_obs, annot=True, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, fmt='.3f', cbar_kws={'label': 'Pearson correlation'})
plt.title('Correlation between CV scores and marker genes scores')
plt.xticks(rotation=45, ha='right')   # horizontal alignment
plt.yticks(rotation=0) 
plt.tight_layout()
plt.show()

In [ ]:
corr_matrix_obs = TCR_embedings.pearson_corr(mdata[test_mask]['gex'], cv_obs_name, scores, var_type='obs')


In [ ]:
# corr_matrix_obs.loc['CV_score_1', 'SPL_score']=0.0964

In [ ]:

plt.figure(figsize=(8, 3))
# Reorder columns of the correlation matrix to match the order of 'scores'
corr_matrix_obs = corr_matrix_obs[scores]

sns.heatmap(corr_matrix_obs, annot=True, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, fmt='.3f', cbar_kws={'label': 'Pearson correlation'})
plt.title('Correlation between CV scores and marker genes scores')
plt.xticks(rotation=45, ha='right')   # horizontal alignment
plt.yticks(rotation=0) 
plt.tight_layout()
plt.show()

In [ ]:
# LEE annotation markers
# marker_genes = ['Cd4','Cd8a', 'Foxp3', 'Ccr6', 'Il17', 'Isg15', 'Icos', 'Pdcd1', 'Ccr7']

# Ref markers
marker_genes = ['Cd4','Cd8a', 'Cd8b1','Cd28', 'Foxp3', 'Ccr6', 'Ptprc', 'Ikzf2', 'Rtkn2', 'Klrg1', 'Isg15', 'Icos', 'Pdcd1', 'Ccr7']

corr_matrix_genes = TCR_embedings.pearson_corr(mdata['gex'], cv_obs_name, marker_genes, var_type='gene')

plt.figure(figsize=(8, 3))
sns.heatmap(corr_matrix_genes, annot=True, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, fmt='.3f', cbar_kws={'label': 'Pearson correlation'})
plt.title('Correlation between CV scores and gene set scores')
plt.xticks(rotation=45, ha='right')   # horizontal alignment
plt.yticks(rotation=0) 
plt.tight_layout()
plt.show()

# Save

In [ ]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True
mdata.write(f'{filename}_singlets_CVscores.h5mu')

In [ ]:
mdata